In [1]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from scipy.stats import zscore
from shapely.geometry import Polygon, Point

import geopandas as gpd
import h3

# reset working dir
import os
from pathlib import Path

In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/anthony/Documents/Dokumente – MacBook Pro von Anthony/UNI/AAA/AAA_TA_2026


In [3]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load cleaned data                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

weather = pd.read_csv("data/weather/01_04_weather_clean.csv")

trips= pd.read_csv("data/trips/01_04_trips_clean.csv")

In [4]:
type(weather["date"][6])
print(weather["date"][6])

2025-01-01 06:00:00


Create merged set without removing weather outliers:

In [5]:
# # # # # # # # # # # # # # # #
#                             #
# Merge trip and weather data #
#                             #
# # # # # # # # # # # # # # # # 

# check for duplicate timestamps and show them before removing
n_dupes = weather["date"].duplicated().sum()
print(f"Duplicate weather timestamps: {n_dupes}")
if n_dupes > 0:
    print(weather[weather["date"].duplicated(keep=False)].sort_values("date"))
    weather = weather.drop_duplicates(subset="date")

# convert datetime columns to datetime
trips["trip_start_timestamp"] = pd.to_datetime(trips["trip_start_timestamp"])
weather["date"] = pd.to_datetime(weather["date"])

# create an hour-level key on the trip data
trips["trip_start_hour"] = trips["trip_start_timestamp"].dt.floor("h")

# merge on the hour key
df_merged = trips.merge(weather, left_on="trip_start_hour", right_on="date", how="left")
df_merged = df_merged.drop(columns=["date"])

assert len(df_merged) == len(trips), "Row count changed after merge — unexpected duplicates remain!"
print(f"Merged shape: {df_merged.shape}")
print(f"Weather NAs after merge: {df_merged['temperature_2m'].isna().sum()}")
df_merged.head()

Duplicate weather timestamps: 1
      index                 date  temperature_2m  relative_humidity_2m  \
7320  16110  2025-11-02 01:00:00             4.2             62.176895   
7321  16111  2025-11-02 01:00:00             4.0             62.128395   

      apparent_temperature  precipitation  rain  snowfall  snow_depth  \
7320              0.444203            0.0   0.0       0.0         0.0   
7321              0.165052            0.0   0.0       0.0         0.0   

      surface_pressure  cloud_cover  wind_speed_10m  wind_speed_100m  is_day  \
7320         887.82780          0.0        8.836514         4.889908     0.0   
7321         888.17206          0.0        9.199390         6.310277     0.0   

      sunshine_duration  direct_radiation  
7320                0.0               0.0  
7321                0.0               0.0  
Merged shape: (5637179, 45)
Weather NAs after merge: 0


,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,rain,snowfall,snow_depth,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation
0,0275e2d8147a31e1ce320c5fb15f9910563cafe1,84957c8960b674346784746bbc1d48cafff4976b162323...,2025-01-01,2025-01-01 00:15:00,758.0,2.93,1.703108e+10,1.703108e+10,8.0,8.0,...,0.0,0.0,0.01,887.8841,0.0,6.927077,4.32,0.0,0.0,0.0
1,05aa05bf6f3ec476715fad9f706bd137e08e00b7,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,2025-01-01,2025-01-01 00:15:00,1233.0,13.66,1.703176e+10,1.703106e+10,76.0,6.0,...,0.0,0.0,0.01,887.8841,0.0,6.927077,4.32,0.0,0.0,0.0
2,17365c83264f028a307ac70308d770fe03bcbcae,c3f8e0b6712bf3ea80e75ddde065b0ed42aa530e8c40cb...,2025-01-01,2025-01-01 00:15:00,985.0,3.22,1.703108e+10,1.703107e+10,8.0,7.0,...,0.0,0.0,0.01,887.8841,0.0,6.927077,4.32,0.0,0.0,0.0
3,1c629303d7f08492d97e13fabe6a0d1d81e6c9c5,c8f57a1150c210a9e6b3fcfb24c3d6d0a43d1879b4b979...,2025-01-01,2025-01-01 00:15:00,652.0,3.24,1.703122e+10,1.703107e+10,22.0,7.0,...,0.0,0.0,0.01,887.8841,0.0,6.927077,4.32,0.0,0.0,0.0
4,1cb48762978a475116ec833c0e4de8ed2ac88e14,88d8896a85cf755c4fb03711d495dc47ca8109196cdef6...,2025-01-01,2025-01-01 00:15:00,803.0,6.09,1.703108e+10,1.703103e+10,8.0,3.0,...,0.0,0.0,0.01,887.8841,0.0,6.927077,4.32,0.0,0.0,0.0


In [6]:
df_merged.isna().sum().to_frame(name="Null Count").assign(
    Null_Percent=lambda x: (x["Null Count"] / len(df_merged) * 100).round(2)
)

,Null Count,Null_Percent
trip_id,0,0.0
taxi_id,0,0.0
trip_start_timestamp,0,0.0
trip_end_timestamp,0,0.0
trip_seconds,0,0.0
trip_miles,0,0.0
pickup_census_tract,0,0.0
dropoff_census_tract,0,0.0
pickup_community_area,0,0.0
dropoff_community_area,0,0.0


In [ ]:
os.makedirs("/data/merged", exist_ok=True)
df_merged.to_csv("data/merged/01_05_trips_weather_merged.csv", index=None)